# 🇻🇳 Speech-MASSIVE Vietnamese Tool-Calling Dataset Generator
**Source:** `doof-ferb/Speech-MASSIVE_vie` → Sinh response → Push lên HuggingFace  
**Model:** `unsloth/gemma-4-E4B-it-unsloth-bnb-4bit`
**Output:** Dataset sẵn sàng cho LoRA training (Giữ nguyên cột Audio)

In [ ]:
# @title 1. Cài đặt
!pip install -q --upgrade "transformers>=4.52" accelerate bitsandbytes datasets huggingface_hub

import transformers
print(f"✅ transformers=={transformers.__version__}")

In [ ]:
# @title 2. Kiểm tra GPU
import torch
print(f"CUDA: {torch.cuda.is_available()} | GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

In [ ]:
# @title 3. Đăng nhập HuggingFace
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# @title 4. Tải dataset & Mapping
from datasets import load_dataset
import json

ds = load_dataset("doof-ferb/Speech-MASSIVE_vie")

TOOL_INTENTS = {
    "alarm_query", "alarm_remove", "alarm_set",
    "audio_volume_down", "audio_volume_mute", "audio_volume_other", "audio_volume_up",
    "calendar_query", "calendar_remove", "calendar_set",
    "cooking_query", "cooking_recipe",
    "datetime_convert", "datetime_query",
    "email_addcontact", "email_query", "email_querycontact", "email_sendemail",
    "iot_cleaning", "iot_coffee",
    "iot_hue_lightchange", "iot_hue_lightdim", "iot_hue_lightoff", "iot_hue_lighton", "iot_hue_lightup",
    "iot_wemo_off", "iot_wemo_on",
    "lists_createoradd", "lists_query", "lists_remove",
    "music_dislikeness", "music_likeness", "music_query", "music_settings",
    "news_query",
    "play_audiobook", "play_game", "play_music", "play_podcasts", "play_radio",
    "qa_currency", "qa_definition", "qa_factoid", "qa_maths", "qa_stock",
    "recommendation_events", "recommendation_locations", "recommendation_movies",
    "social_post", "social_query",
    "takeaway_order", "takeaway_query",
    "transport_query", "transport_taxi", "transport_ticket", "transport_traffic",
    "weather_query",
}
GENERAL_INTENTS = {"general_greet", "general_joke", "general_quirky"}

def build_tool_call(intent, utt):
    if intent in GENERAL_INTENTS or intent not in TOOL_INTENTS:
        return None
    return {"name": intent, "arguments": {"query": utt}}

In [ ]:
# @title 5. Nạp Model
import torch
import gc
from transformers import AutoProcessor, Gemma4ForConditionalGeneration, BitsAndBytesConfig, AutoTokenizer, AutoModelForCausalLM

torch.cuda.empty_cache()
gc.collect()

REPO = "unsloth/gemma-4-E4B-it-unsloth-bnb-4bit"
print(f"Loading {REPO}...")

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

try:
    processor = AutoProcessor.from_pretrained(REPO)
    model = Gemma4ForConditionalGeneration.from_pretrained(
        REPO, 
        device_map="auto",
        quantization_config=quantization_config,
        torch_dtype=torch.float16,
    )
    IS_MULTIMODAL = True
    print("✅ Loaded as Multimodal")
except Exception as e:
    print(f"⚠️ Fallback to CausalLM... Error: {e}")
    processor = AutoTokenizer.from_pretrained(REPO)
    model = AutoModelForCausalLM.from_pretrained(
        REPO, 
        device_map="auto",
        quantization_config=quantization_config,
        torch_dtype=torch.float16,
    )
    IS_MULTIMODAL = False
    print("✅ Loaded as Text")

model.eval()
print(f"✅ GPU VRAM used: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

In [ ]:
# @title 6. Hàm Sinh với Chat Template Chuẩn
import time
import gc

BATCH_SIZE = 1  # Tăng lên 2 hoặc 4 nếu VRAM cho phép

def make_messages(utt, intent, scenario, tool_hint_str):
    sys_msg = (
        "Bạn là trợ lý ảo tiếng Việt thông minh và lịch sự.\n"
        "Nhiệm vụ: Phản hồi câu nói của người dùng.\n\n"
        "ĐỊNH DẠNG TRẢ LỜI BẮT BUỘC:\n"
        "- NẾU CẦN CÔNG CỤ: Bắt đầu bằng <tool_call>{\"name\": \"tên\", \"arguments\": {\"tham_số\": \"giá_trị\"}}</tool_call> sau đó xuống dòng và viết câu trả lời tiếng Việt.\n"
        "- NẾU KHÔNG CẦN CÔNG CỤ: Chỉ viết câu trả lời tiếng Việt tự nhiên, tuyệt đối không dùng thẻ <tool_call>.\n\n"
        "Yêu cầu:\n"
        "- Giọng điệu tự nhiên, lịch sự (dạ, nhé, ạ).\n"
        "- Trả lời trực tiếp vào yêu cầu, không lặp lại prompt."
    )
    user_msg = (
        f"Câu nói: {utt}\n"
        f"Ý định: {intent}\n"
        f"Bối cảnh: {scenario}\n"
        f"Gợi ý công cụ: {tool_hint_str}"
    )
    return [
        {"role": "user", "content": f"{sys_msg}\n\n{user_msg}"}
    ]

def apply_chat_template_safe(messages):
    if hasattr(processor, "apply_chat_template"):
        return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    elif hasattr(processor, "tokenizer") and hasattr(processor.tokenizer, "apply_chat_template"):
        return processor.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        return f"<start_of_turn>user\n{messages[0]['content']}<end_of_turn>\n<start_of_turn>model\n"

def generate_batch(batch_messages):
    prompts = [apply_chat_template_safe(msg) for msg in batch_messages]
    
    if IS_MULTIMODAL:
        inputs = processor(text=prompts, return_tensors="pt", padding=True).to("cuda:0")
    else:
        processor.padding_side = "left"
        if not processor.pad_token:
            processor.pad_token = processor.eos_token
        inputs = processor(prompts, return_tensors="pt", padding=True).to("cuda:0")
    
    input_length = inputs["input_ids"].shape[1]
    
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.1, top_p=0.9,
            do_sample=True, repetition_penalty=1.1,
        )
    
    results = []
    for i in range(len(prompts)):
        # Chỉ lấy phần generated tokens (bỏ qua phần prompt ban đầu)
        gen_tokens = out[i][input_length:]
        text = processor.decode(gen_tokens, skip_special_tokens=True).strip()
        results.append(text)
    
    del inputs, out
    torch.cuda.empty_cache()
    return results

# Dry run
t0 = time.time()
msgs = [make_messages("đặt báo thức lúc 6 giờ sáng", "alarm_set", "alarm", '{"name":"alarm_set","arguments":{"time":"06:00"}}')]
resps = generate_batch(msgs)
print(f"({time.time()-t0:.1f}s) → {resps[0]}")

In [ ]:
# @title 7. 🚀 Sinh response
import json, time, gc, os

CKPT_DIR = "/content/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

all_results = {}

for split_name in ["train", "validation", "test"]:
    split_data = ds[split_name]
    ckpt_file = f"{CKPT_DIR}/{split_name}.json"

    results = []
    done_ids = set()
    if os.path.exists(ckpt_file):
        with open(ckpt_file, "r") as f:
            results = json.load(f)
            done_ids = {str(r["id"]) for r in results}
        print(f"🔄 [{split_name}] Resume: {len(results)} done")

    remaining = [row for row in split_data if str(row["id"]) not in done_ids]
    total = len(remaining)
    print(f"\n{'='*50}")
    print(f"[{split_name}] Total: {len(split_data)} | Remaining: {total}")
    print(f"{'='*50}")

    if total == 0:
        all_results[split_name] = results
        continue

    start = time.time()
    ckpt_n = 0

    for batch_start in range(0, total, BATCH_SIZE):
        batch = remaining[batch_start:batch_start + BATCH_SIZE]

        batch_msgs = []
        tool_calls = []
        for row in batch:
            tc = build_tool_call(row["intent_str"], row["utt"])
            tool_calls.append(tc)
            hint = json.dumps(tc, ensure_ascii=False) if tc else "Không có (General)"
            batch_msgs.append(make_messages(row["utt"], row["intent_str"], row["scenario_str"], hint))

        try:
            resps = generate_batch(batch_msgs)
        except Exception as e:
            print(f"\n❌ Lỗi Batch: {e}")
            torch.cuda.empty_cache()
            continue

        for i, row in enumerate(batch):
            resp = resps[i]
            tc = tool_calls[i]
            if tc and "<tool_call>" not in resp:
                resp = f'<tool_call>{json.dumps(tc, ensure_ascii=False)}</tool_call>\n' + resp
            results.append({"id": str(row["id"]), "utt": row["utt"], "intent": row["intent_str"],
                            "scenario": row["scenario_str"], "response": resp})

        done = batch_start + len(batch)
        ckpt_n += len(batch)
        elapsed = time.time() - start
        speed = elapsed / done
        print(f"\r[{split_name}] {done}/{total} | {speed:.1f}s/sample | ETA: {speed*(total-done)/3600:.1f}h", end="", flush=True)

        if ckpt_n >= 50 or done >= total:
            with open(ckpt_file, "w") as f:
                json.dump(results, f, ensure_ascii=False, indent=2)
            ckpt_n = 0
            print(f"\n💾 [{split_name}] {len(results)} saved")

        if done % 100 == 0:
            torch.cuda.empty_cache(); gc.collect()

    all_results[split_name] = results
    print(f"\n✅ [{split_name}] {len(results)} in {(time.time()-start)/3600:.2f}h")

print("\n🎉 Done!")

In [ ]:
# @title 8. Giữ nguyên cột Audio & Push lên HuggingFace
from datasets import DatasetDict

# 1. Xây dựng dictionary ánh xạ id -> response
resp_dict = {str(r["id"]): r["response"] for split_res in all_results.values() for r in split_res}

# 2. Hàm ánh xạ để CHỈ THÊM các cột mới vào dataset gốc (giữ nguyên 'audio')
def add_lora_columns(example):
    sample_id = str(example["id"])
    response = resp_dict.get(sample_id, "")
    
    instruction = f"Phản hồi câu nói dưới dạng tool call hoặc trả lời tự nhiên.\nÝ định: {example['intent_str']}\nBối cảnh: {example['scenario_str']}"
    
    return {
        "instruction": instruction,
        "input": example["utt"],
        "output": response,
    }

# 3. Gọi map trên dataset gốc (`ds`) để giữ trọn vẹn Audio và tất cả các feature khác
lora_ds = ds.map(add_lora_columns)
print(lora_ds)

# 4. Upload
# === SỬA TÊN REPO CỦA BẠN ===
HF_REPO = "YOUR_USERNAME/speech-massive-vie-tool-calling"

lora_ds.push_to_hub(
    HF_REPO, 
    private=True, 
    commit_message="Gemma 4 E4B generated Vietnamese tool-calling WITH AUDIO"
)
print(f"\n🎉 Pushed: https://huggingface.co/datasets/{HF_REPO}")